In [ ]:
# ============================================================
# The Metals Company — Press Releases Scraper
# Compatible with Google Colab
# ============================================================
# Scrapes: https://investors.metals.co/news-events/press-releases
# Output:  press_releases.csv  (Title, Date, URL)
# ============================================================

# --- 1. Install dependencies (Colab may already have these) ---
# Uncomment the line below if running for the first time in Colab:
# !pip install requests beautifulsoup4 --quiet

import requests
from bs4 import BeautifulSoup
import csv
import time

# --- 2. Configuration ---
BASE_URL = "https://investors.metals.co/news-events/press-releases"
OUTPUT_FILE = "press_releases.csv"
DELAY_SECONDS = 1  # Polite delay between requests

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

# --- 3. Helper: get the total number of pages ---
def get_total_pages(session):
    response = session.get(BASE_URL, headers=HEADERS)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Find the "Last page" pagination link, e.g. ?page=14 → 15 pages
    last_link = soup.select_one('a[title="Go to last page"]')
    if last_link:
        href = last_link.get("href", "")
        # href looks like /news-events/press-releases?page=14
        last_page_index = int(href.split("page=")[-1])
        return last_page_index + 1  # 0-indexed, so add 1
    return 1  # Fallback: only one page

# --- 4. Helper: scrape one page ---
def scrape_page(session, page_index):
    url = f"{BASE_URL}?page={page_index}"
    response = session.get(url, headers=HEADERS)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    records = []

    # Each press release is an <h5> (or <h4>/<h3>) containing an <a> tag
    # The date follows as a sibling text node or a <p>/<span> nearby.
    # Structure observed:
    #   <h5><a href="...">Title</a></h5>
    #   <p>Date</p>   OR plain text node

    # Select all article/release title links
    # The site wraps each entry in an <article> or uses h5 > a pattern
    entries = soup.select("h5 a, h4 a, h3 a")  # cast a wide net

    for a_tag in entries:
        title = a_tag.get_text(strip=True)
        href = a_tag.get("href", "")

        # Build absolute URL if relative
        if href.startswith("/"):
            href = "https://investors.metals.co" + href
        elif not href.startswith("http"):
            continue  # skip nav/footer links without a real path

        # Skip navigation links (too short or clearly not a press release)
        if len(title) < 10:
            continue

        # Find the date — look in the parent element's siblings
        parent = a_tag.find_parent(["h3", "h4", "h5"])
        date_text = ""
        if parent:
            # Date is often the next sibling element
            sibling = parent.find_next_sibling()
            if sibling:
                date_text = sibling.get_text(strip=True)

        records.append({
            "Title": title,
            "Date": date_text,
            "URL": href,
        })

    return records

# --- 5. Main scraper ---
def main():
    all_records = []

    with requests.Session() as session:
        print("Detecting total pages...")
        total_pages = get_total_pages(session)
        print(f"Found {total_pages} page(s) to scrape.\n")

        for page_index in range(total_pages):
            print(f"  Scraping page {page_index + 1} of {total_pages}...", end=" ")
            try:
                records = scrape_page(session, page_index)
                all_records.extend(records)
                print(f"{len(records)} releases found.")
            except requests.RequestException as e:
                print(f"ERROR — {e}")
            time.sleep(DELAY_SECONDS)

    # --- 6. Write to CSV ---
    print(f"\nWriting {len(all_records)} records to '{OUTPUT_FILE}'...")
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["Title", "Date", "URL"])
        writer.writeheader()
        writer.writerows(all_records)

    print(f"Done! Saved to: {OUTPUT_FILE}")

    # --- 7. Preview first 5 rows (Colab-friendly) ---
    try:
        import pandas as pd
        df = pd.read_csv(OUTPUT_FILE)
        print(f"\nTotal rows: {len(df)}")
        print(df.head())
    except ImportError:
        pass  # pandas not required

if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# The Metals Company — Press Release CONTENT Scraper (Script 2)
# Compatible with Google Colab — paste into a new code cell
# ============================================================
# Prerequisites: Run Script 1 first so press_releases.csv exists
# Input:  press_releases.csv  (Title, Date, URL)
# Output: press_releases_content.csv  (Title, Date, URL, Content)
# ============================================================

# --- 1. Install / verify dependencies ---
# Uncomment if needed (usually pre-installed in Colab):
# !pip install requests beautifulsoup4 --quiet

import requests
from bs4 import BeautifulSoup
import csv
import time
import os

# --- 2. Configuration ---
INPUT_FILE  = "press_releases.csv"
OUTPUT_FILE = "press_releases_content.csv"
DELAY_SECONDS = 1.2   # Polite pause between requests

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

# --- 3. Tags and strings to strip from the page before extracting text ---
# These selectors match navigation, headers, footers, and other chrome.
STRIP_SELECTORS = [
    "nav", "header", "footer",
    ".menu", ".breadcrumb", ".pagination",
    "#main-menu", "#main-navigation",
    "[role='navigation']", "[role='banner']",
    "script", "style", "noscript",
]

# Footer boilerplate phrases — content is cut off at the first match
BOILERPLATE_CUTOFFS = [
    "We're building a carefully managed metal commons",
    "Privacy Policy",
]

# --- 4. Core extraction function ---
def extract_content(html: str) -> tuple[str, str, str]:
    """
    Returns (title, date, body_text) extracted from a press release page.
    Falls back gracefully if elements are missing.
    """
    soup = BeautifulSoup(html, "html.parser")

    # -- Title --
    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else ""

    # -- Date --
    # Pattern on this site: "by  The Metals Company |\\n May 14, 2026"
    # It usually lives in a <p> or sibling near the h1.
    date = ""
    if h1:
        for sibling in h1.find_next_siblings():
            text = sibling.get_text(" ", strip=True)
            # Look for a sibling that contains a month name
            months = ["January","February","March","April","May","June",
                      "July","August","September","October","November","December"]
            if any(m in text for m in months):
                # Strip the "by The Metals Company |" prefix if present
                if "|" in text:
                    date = text.split("|", 1)[-1].strip()
                else:
                    date = text.strip()
                break

    # -- Body content --
    # Remove navigation/chrome elements in-place
    for selector in STRIP_SELECTORS:
        for tag in soup.select(selector):
            tag.decompose()

    # Try Drupal-common content wrappers first, then fall back to <main> / <body>
    content_candidates = [
        soup.select_one(".field--name-body"),
        soup.select_one(".node__content"),
        soup.select_one("article"),
        soup.select_one("main"),
        soup.find("body"),
    ]
    content_el = next((el for el in content_candidates if el), None)

    if content_el:
        # Remove the h1 and date paragraph from the content block
        for tag in content_el.find_all(["h1"]):
            tag.decompose()

        raw_text = content_el.get_text(separator="\n", strip=True)
    else:
        raw_text = soup.get_text(separator="\n", strip=True)

    # Trim at boilerplate cutoff phrases
    body = raw_text
    for cutoff in BOILERPLATE_CUTOFFS:
        idx = body.find(cutoff)
        if idx != -1:
            body = body[:idx].strip()

    # Collapse excessive blank lines
    lines = body.splitlines()
    cleaned_lines = []
    prev_blank = False
    for line in lines:
        is_blank = line.strip() == ""
        if is_blank and prev_blank:
            continue   # skip consecutive blank lines
        cleaned_lines.append(line)
        prev_blank = is_blank

    body = "\n".join(cleaned_lines).strip()

    return title, date, body


# --- 5. Main routine ---
def main():
    # Check input file exists
    if not os.path.exists(INPUT_FILE):
        print(f"ERROR: '{INPUT_FILE}' not found.")
        print("Please run Script 1 first to generate the press releases list.")
        return

    # Load the list of press releases
    with open(INPUT_FILE, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    total = len(rows)
    print(f"Loaded {total} press releases from '{INPUT_FILE}'.")
    print(f"Starting content scrape — estimated time: ~{round(total * DELAY_SECONDS / 60, 1)} minutes\n")

    results = []
    errors  = []

    with requests.Session() as session:
        for i, row in enumerate(rows, start=1):
            url   = row.get("URL", "").strip()
            title_fallback = row.get("Title", "").strip()
            date_fallback  = row.get("Date",  "").strip()

            if not url:
                print(f"  [{i}/{total}] SKIP — no URL")
                continue

            print(f"  [{i}/{total}] {title_fallback[:70]}...", end=" ")

            try:
                resp = session.get(url, headers=HEADERS, timeout=20)
                resp.raise_for_status()

                title, date, content = extract_content(resp.text)

                # Use CSV fallbacks if extraction came up empty
                if not title:
                    title = title_fallback
                if not date:
                    date = date_fallback

                results.append({
                    "Title":   title,
                    "Date":    date,
                    "URL":     url,
                    "Content": content,
                })
                print(f"OK ({len(content)} chars)")

            except requests.RequestException as e:
                print(f"ERROR — {e}")
                errors.append({"url": url, "error": str(e)})
                # Still write a row so the CSV stays aligned
                results.append({
                    "Title":   title_fallback,
                    "Date":    date_fallback,
                    "URL":     url,
                    "Content": "ERROR: could not fetch page",
                })

            time.sleep(DELAY_SECONDS)

    # --- 6. Write output CSV ---
    print(f"\nWriting {len(results)} records to '{OUTPUT_FILE}'...")
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["Title", "Date", "URL", "Content"])
        writer.writeheader()
        writer.writerows(results)

    print(f"Done! Saved to: {OUTPUT_FILE}")

    if errors:
        print(f"\nWarning: {len(errors)} URL(s) failed to load:")
        for e in errors:
            print(f"  - {e['url']}: {e['error']}")

    # --- 7. Preview first 3 rows (Colab-friendly) ---
    try:
        import pandas as pd
        df = pd.read_csv(OUTPUT_FILE)
        print(f"\nTotal rows: {len(df)}")
        # Show title + first 200 chars of content for preview
        preview = df[["Title", "Date", "URL"]].copy()
        preview["Content (preview)"] = df["Content"].str[:200] + "..."
        print(preview.head(3).to_string(index=False))
    except ImportError:
        pass


if __name__ == "__main__":
    main()